In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Cargando dataset completo de 18 días...")
df = pd.read_csv("coldtrack_dataset_completo.csv")

print(f"Dataset cargado: {df.shape[0]:,} registros × {df.shape[1]} columnas")

# ====================== 1. CONVERTIR TIMESTAMP ======================
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format="%A %d.%m.%Y -- %H:%M:%S")
df = df.sort_values('Timestamp').reset_index(drop=True)

print("Creando características temporales avanzadas...")

# 1. Variables temporales
df['hour'] = df['Timestamp'].dt.hour
df['minute'] = df['Timestamp'].dt.minute
df['day_of_week'] = df['Timestamp'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# 2. Diferencia con temperatura ideal (setpoint = 2.5°C)
df['temp_diff_setpoint'] = df['inTemp'] - 2.5

# 3. Media móvil y desviación estándar (ventanas importantes)
windows = {
    '5min': 60,      # 5 minutos
    '15min': 180,    # 15 minutos
    '30min': 360     # 30 minutos
}

for col in ['inTemp', 'Vibration', 'ConsumoElectrico']:
    for name, w in windows.items():
        df[f'{col}_ma_{name}'] = df[col].rolling(window=w, min_periods=1).mean()
        if col != 'ConsumoElectrico':  # std no es tan útil en consumo
            df[f'{col}_std_{name}'] = df[col].rolling(window=w, min_periods=1).std()

# 4. Tiempo acumulado de puerta abierta en la última hora
df['DoorOPEN_flag'] = (df['DoorStatus'] == 'DoorOPEN').astype(int)
df['door_open_last_hour'] = df['DoorOPEN_flag'].rolling(window=720, min_periods=1).sum()

# 5. Tendencia de temperatura (slope) en los últimos 30 minutos
def calculate_slope(x):
    if len(x) < 2:
        return 0.0
    t = np.arange(len(x))
    slope = np.polyfit(t, x, 1)[0]
    return slope

df['inTemp_trend_30min'] = df['inTemp'].rolling(window=360, min_periods=10).apply(calculate_slope, raw=True)

# 6. Features derivadas de riesgo
df['high_temp'] = (df['inTemp'] > 5.5).astype(int)
df['critical_vibration'] = (df['Vibration'] > 0.8).astype(int)
df['long_door_open'] = (df['door_open_last_hour'] > 20).astype(int)
df['compressor_overwork'] = (df['ConsumoElectrico'] > 180).astype(int)

# ====================== 7. SELECCIÓN FINAL DE FEATURES ======================
feature_columns = [
    'inTemp', 'InHumid', 'outTemp', 'outHumid', 'ConsumoElectrico', 'Vibration',
    'temp_diff_setpoint',
    'inTemp_ma_5min', 'inTemp_ma_15min', 'inTemp_ma_30min',
    'inTemp_std_5min', 'inTemp_std_15min',
    'Vibration_ma_5min', 'Vibration_ma_15min', 'Vibration_std_5min',
    'ConsumoElectrico_ma_5min',
    'door_open_last_hour',
    'inTemp_trend_30min',
    'high_temp', 'critical_vibration', 'long_door_open', 'compressor_overwork',
    'hour', 'day_of_week', 'is_weekend'
]

X = df[feature_columns].copy()
y = df['flag_mantenimiento'].copy()

print("\n" + "="*70)
print("FEATURE ENGINEERING FINALIZADO")
print("="*70)
print(f"Features creadas: {X.shape[1]} columnas")
print(f"Registros totales: {X.shape[0]:,}")
print(f"Distribución de clases:\n{y.value_counts(normalize=True).round(4)*100}")

# Guardar datasets procesados
X.to_csv("X_features.csv", index=False)
y.to_csv("y_target.csv", index=False)
pd.concat([X, y], axis=1).to_csv("coldtrack_features_target.csv", index=False)

print("\n✅ Archivos guardados:")
print("   • X_features.csv")
print("   • y_target.csv")
print("   • coldtrack_features_target.csv  ← (este usaremos para entrenar)")

print("\nListo para entrenar el Random Forest.")

Cargando dataset completo de 18 días...
Dataset cargado: 311,040 registros × 12 columnas
Creando características temporales avanzadas...

FEATURE ENGINEERING FINALIZADO
Features creadas: 25 columnas
Registros totales: 311,040
Distribución de clases:
flag_mantenimiento
0    92.03
1     7.97
Name: proportion, dtype: float64

✅ Archivos guardados:
   • X_features.csv
   • y_target.csv
   • coldtrack_features_target.csv  ← (este usaremos para entrenar)

Listo para entrenar el Random Forest.
